# Lineare Regression mit Mitteln der linearen Algebra

**Anwendungen der linearen Algebra — Probevorlesung**

Dieses Notebook begleitet die Vorlesung. Wir erarbeiten Schritt für Schritt:

1. Messdaten und das überbestimmte Gleichungssystem
2. Die Designmatrix $A$
3. Die Normalengleichung und ihre Lösung
4. Visualisierung: Regressionsgerade und Residuen
5. Verifikation: $A^\top \mathbf{e} \approx \mathbf{0}$
6. Erweiterung: Polynomregression
7. Erweiterung: Multiple lineare Regression

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Reproduzierbarkeit
np.random.seed(42)

# Matplotlib-Stil
plt.rcParams.update({
    'font.size': 12,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

---
## 1. Messdaten generieren

Wir erzeugen synthetische Messdaten mit einem linearen Zusammenhang:

$$y_k = 2 \cdot t_k + 1 + \varepsilon_k, \qquad \varepsilon_k \sim \mathcal{N}(0,\, 0.5^2)$$

Das «wahre» Modell hat also $m = 2$ und $b = 1$. Unser Ziel: diese Parameter aus den verrauschten Daten rekonstruieren.

In [ ]:
# Parameter des wahren Modells
m_true, b_true = 2.0, 1.0
sigma_noise = 0.5

# Datenpunkte
n = 20
t = np.linspace(0.5, 5.5, n)
noise = np.random.normal(0, sigma_noise, n)
y = m_true * t + b_true + noise

# Visualisierung
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(t, y, color='steelblue', s=60, zorder=5, label='Messdaten $(t_k, y_k)$')
ax.set_xlabel('$t$')
ax.set_ylabel('$y$')
ax.set_title('Messdaten mit linearem Trend')
ax.legend()
plt.tight_layout()
plt.show()

print(f"Anzahl Messpunkte: n = {n}")
print(f"Wahres Modell:     y = {m_true}·t + {b_true}  (+ Rauschen σ={sigma_noise})")

---
## 2. Designmatrix $A$ aufstellen

Wir suchen $\hat{\mathbf{x}} = (m, b)^\top$, so dass $A\hat{\mathbf{x}} \approx \mathbf{y}$.

Jeder Messpunkt liefert eine Gleichung:
$$m \cdot t_k + b \cdot 1 = y_k$$

Die **Designmatrix** $A \in \mathbb{R}^{n \times 2}$ hat in Zeile $k$: $[t_k,\; 1]$.

In [ ]:
# Designmatrix aufstellen
# Spalte 1: t-Werte  |  Spalte 2: Einsen (für den y-Achsenabschnitt b)
A = np.column_stack([t, np.ones(n)])

print("Designmatrix A (erste 5 Zeilen):")
print("  t_k    1")
print("-" * 14)
for row in A[:5]:
    print(f"  {row[0]:.2f}   {row[1]:.0f}")
print("  ...")
print(f"\nForm von A: {A.shape}  →  {A.shape[0]} Gleichungen, {A.shape[1]} Unbekannte")

---
## 3. Normalengleichung lösen

Das System $A\hat{\mathbf{x}} = \mathbf{y}$ ist überbestimmt — keine exakte Lösung.

**Geometrische Idee:** Projiziere $\mathbf{y}$ auf $\operatorname{Bild}(A)$.
Der Fehlervektor $\mathbf{e} = \mathbf{y} - \hat{\mathbf{y}}$ steht senkrecht auf $\operatorname{Bild}(A)$,
also $\mathbf{e} \in \operatorname{Kern}(A^\top)$, also $A^\top \mathbf{e} = \mathbf{0}$.

Das führt zur **Normalengleichung**:
$$A^\top A \,\hat{\mathbf{x}} = A^\top \mathbf{y}
\qquad\Longrightarrow\qquad
\hat{\mathbf{x}} = (A^\top A)^{-1} A^\top \mathbf{y}$$

In [ ]:
# Normalengleichung: A^T A x_hat = A^T y
ATA = A.T @ A          # (2×n)(n×2) = 2×2
ATy = A.T @ y          # (2×n)(n,)  = 2,

# Lösung über np.linalg.solve (numerisch stabil)
x_hat = np.linalg.solve(ATA, ATy)
m_hat, b_hat = x_hat

print("A^T A =\n", ATA)
print("\nA^T y =\n", ATy)
print()
print(f"Geschätzte Parameter:  m̂ = {m_hat:.4f},  b̂ = {b_hat:.4f}")
print(f"Wahre Parameter:       m  = {m_true:.4f},  b  = {b_true:.4f}")

---
## 4. Visualisierung: Regressionsgerade und Residuen

Die **Residuen** (Fehler) $e_k = y_k - \hat{y}_k$ zeigen, wie weit jeder Messpunkt von der
Regressionsgeraden entfernt ist.

In [ ]:
# Vorhergesagte Werte
y_hat = A @ x_hat
e = y - y_hat

# Regressionsgerade
t_fit = np.linspace(0, 6, 200)
y_fit = m_hat * t_fit + b_hat

fig, ax = plt.subplots(figsize=(9, 5.5))

# Residuen
for ti, yi, yhi in zip(t, y, y_hat):
    ax.plot([ti, ti], [yi, yhi], color='tomato', linewidth=1.2, alpha=0.8, zorder=3)

# Messpunkte
ax.scatter(t, y, color='steelblue', s=60, zorder=5, label='Messdaten $(t_k, y_k)$')

# Projektionen auf der Geraden
ax.scatter(t, y_hat, color='steelblue', s=25, marker='D', zorder=4, alpha=0.5,
           label='Projektionen $\\hat{y}_k$')

# Regressionsgerade
ax.plot(t_fit, y_fit, color='steelblue', linewidth=2.2, zorder=4,
        label=f'Regressionsgerade: $\\hat{{m}}={m_hat:.2f},\;\\hat{{b}}={b_hat:.2f}$')

residual_patch = mpatches.Patch(color='tomato', alpha=0.8, label='Residuen $e_k = y_k - \\hat{y}_k$')
ax.legend(handles=ax.get_legend_handles_labels()[0] + [residual_patch], fontsize=10)

ax.set_xlabel('$t$')
ax.set_ylabel('$y$')
ax.set_title('Lineare Regression: Regressionsgerade und Residuen')
ax.set_xlim(0, 6)
plt.tight_layout()
plt.show()

print(f"Summe der Fehlerquadrate: ||e||² = {e @ e:.4f}")

---
## 5. Verifikation: $A^\top \mathbf{e} \approx \mathbf{0}$

Die Normalengleichung garantiert, dass der Fehlervektor $\mathbf{e}$ im $\operatorname{Kern}(A^\top)$ liegt,
also $A^\top \mathbf{e} = \mathbf{0}$.

Prüfen wir das numerisch:

In [ ]:
ATe = A.T @ e

print("A^T e =", ATe)
print(f"||A^T e|| = {np.linalg.norm(ATe):.2e}")
print()
print("✓ A^T e ist (numerisch) der Nullvektor — e steht senkrecht auf Bild(A).")

### Geometrische Perspektive: Residuen und ihre Summe

Zwei anschauliche Konsequenzen von $A^\top \mathbf{e} = \mathbf{0}$:

- **Spalte 1** ($A^\top \mathbf{e}$, Eintrag 1): $\sum_k t_k e_k = 0$ — die gewichteten Residuen summieren sich zu null
- **Spalte 2** ($A^\top \mathbf{e}$, Eintrag 2): $\sum_k e_k = 0$ — die Residuen heben sich auf

In [ ]:
print(f"Summe der Residuen:              Σ e_k = {e.sum():.2e}   (≈ 0)")
print(f"Gewichtete Summe der Residuen: Σ t_k·e_k = {(t * e).sum():.2e}   (≈ 0)")

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.bar(range(n), e, color=['tomato' if ei > 0 else 'steelblue' for ei in e], alpha=0.8)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_xlabel('Messpunkt $k$')
ax.set_ylabel('$e_k = y_k - \\hat{y}_k$')
ax.set_title('Residuen (positive und negative heben sich auf)')
plt.tight_layout()
plt.show()

---
## 6. Erweiterung: Polynomregression

Was ändert sich, wenn wir ein **quadratisches Modell** verwenden?

$$y = a_2 t^2 + a_1 t + a_0$$

Nur die **Designmatrix** ändert sich — Zeile $k$: $[t_k^2,\; t_k,\; 1]$.  
Die **Normalengleichung** $A^\top A \,\hat{\mathbf{x}} = A^\top \mathbf{y}$ bleibt identisch.

In [ ]:
# Nichtlineare Daten: y = 0.3 t² - 0.5 t + 2 + Rauschen
np.random.seed(7)
t_nl = np.linspace(0.5, 5.5, 25)
y_nl = 0.3 * t_nl**2 - 0.5 * t_nl + 2 + np.random.normal(0, 0.4, 25)

# Designmatrix für quadratisches Modell: Zeile k = [t_k², t_k, 1]
A_poly = np.column_stack([t_nl**2, t_nl, np.ones(len(t_nl))])

print("Designmatrix für Polynomregression (erste 3 Zeilen):")
print("  t_k²    t_k    1")
print("-" * 22)
for row in A_poly[:3]:
    print(f"  {row[0]:5.2f}  {row[1]:5.2f}  {row[2]:.0f}")

# Normalengleichung — exakt gleiche Formel wie vorher!
x_hat_poly = np.linalg.solve(A_poly.T @ A_poly, A_poly.T @ y_nl)
a2, a1, a0 = x_hat_poly
print(f"\nGeschätzte Parameter: â₂ = {a2:.4f},  â₁ = {a1:.4f},  â₀ = {a0:.4f}")
print(f"Wahre Parameter:      a₂ = 0.3000,  a₁ = -0.5000,  a₀ = 2.0000")

In [ ]:
t_fit_nl = np.linspace(0, 6, 300)
y_fit_nl = a2 * t_fit_nl**2 + a1 * t_fit_nl + a0

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Links: lineare Regression
ax = axes[0]
ax.scatter(t, y, color='steelblue', s=50, label='Messdaten')
ax.plot(t_fit, y_fit, color='steelblue', linewidth=2,
        label=f'$\\hat{{m}}={m_hat:.2f},\;\\hat{{b}}={b_hat:.2f}$')
for ti, yi, yhi in zip(t, y, y_hat):
    ax.plot([ti, ti], [yi, yhi], color='tomato', linewidth=1, alpha=0.6)
ax.set_title('Lineare Regression\n(Designmatrix: $[t_k,\\;1]$)')
ax.set_xlabel('$t$'); ax.set_ylabel('$y$'); ax.legend(fontsize=9)

# Rechts: Polynomregression
ax = axes[1]
y_hat_nl = A_poly @ x_hat_poly
ax.scatter(t_nl, y_nl, color='darkorange', s=50, label='Messdaten')
ax.plot(t_fit_nl, y_fit_nl, color='darkorange', linewidth=2,
        label=f'$\\hat{{a}}_2={a2:.2f},\;\\hat{{a}}_1={a1:.2f},\;\\hat{{a}}_0={a0:.2f}$')
for ti, yi, yhi in zip(t_nl, y_nl, y_hat_nl):
    ax.plot([ti, ti], [yi, yhi], color='tomato', linewidth=1, alpha=0.6)
ax.set_title('Polynomregression (Grad 2)\n(Designmatrix: $[t_k^2,\\;t_k,\\;1]$)')
ax.set_xlabel('$t$'); ax.set_ylabel('$y$'); ax.legend(fontsize=9)

plt.suptitle('Gleiche Normalengleichung — nur die Designmatrix ändert sich', fontsize=13)
plt.tight_layout()
plt.show()

---
## 7. Erweiterung: Multiple lineare Regression

Was ändert sich, wenn wir **zwei unabhängige Messgrössen** $u$ und $v$ haben?

$$y = b + a_u\, u + a_v\, v$$

Zeile $k$ der Designmatrix: $[1,\; u_k,\; v_k]$.  
Die **Normalengleichung** $A^\top A \,\hat{\mathbf{x}} = A^\top \mathbf{y}$ bleibt identisch.

In [ ]:
# Daten: y = 1 + 2·u - 0.5·v + Rauschen  (zwei unabhängige Messgrössen)
np.random.seed(3)
n_ml = 30
u_ml = np.random.uniform(0, 5, n_ml)
v_ml = np.random.uniform(0, 5, n_ml)
y_ml = 1.0 + 2.0 * u_ml - 0.5 * v_ml + np.random.normal(0, 0.5, n_ml)

# Designmatrix: Zeile k = [1, u_k, v_k]
A_ml = np.column_stack([np.ones(n_ml), u_ml, v_ml])

print("Designmatrix für multiple Regression (erste 3 Zeilen):")
print("   1    u_k    v_k")
print("-" * 22)
for row in A_ml[:3]:
    print(f"  {row[0]:.0f}   {row[1]:.2f}   {row[2]:.2f}")

# Normalengleichung — exakt gleiche Formel!
x_hat_ml = np.linalg.solve(A_ml.T @ A_ml, A_ml.T @ y_ml)
b_ml, au_ml, av_ml = x_hat_ml
y_hat_ml = A_ml @ x_hat_ml

print(f"\nGeschätzte Parameter: b̂ = {b_ml:.4f},  â_u = {au_ml:.4f},  â_v = {av_ml:.4f}")
print(f"Wahre Parameter:      b = 1.0000,  a_u =  2.0000,  a_v = -0.5000")

In [ ]:
u_fit = np.linspace(0, 5, 50)
v_fit = np.linspace(0, 5, 50)
U_g, V_g = np.meshgrid(u_fit, v_fit)
Y_fit = b_ml + au_ml * U_g + av_ml * V_g

fig = plt.figure(figsize=(12, 5))

# Links: 3D-Darstellung
ax3d = fig.add_subplot(121, projection='3d')
ax3d.plot_surface(U_g, V_g, Y_fit, alpha=0.3, color='darkorange')
ax3d.scatter(u_ml, v_ml, y_ml, color='steelblue', s=40, zorder=5, label='Messdaten')
for k in range(n_ml):
    ax3d.plot([u_ml[k], u_ml[k]], [v_ml[k], v_ml[k]], [y_ml[k], y_hat_ml[k]],
              color='tomato', linewidth=0.8, alpha=0.7)
ax3d.set_xlabel('$u$'); ax3d.set_ylabel('$v$'); ax3d.set_zlabel('$y$')
ax3d.set_title('Multiple Regression\n(Regressionsfläche)')

# Rechts: Vorhergesagt vs. tatsächlich
ax = fig.add_subplot(122)
ax.scatter(y_ml, y_hat_ml, color='darkorange', s=60, zorder=5)
lims = [min(y_ml.min(), y_hat_ml.min()) - 0.5, max(y_ml.max(), y_hat_ml.max()) + 0.5]
ax.plot(lims, lims, 'k--', linewidth=1, alpha=0.5, label='Ideale Vorhersage')
ax.set_xlabel('Tatsächliches $y$')
ax.set_ylabel('Vorhergesagtes $\\hat{y}$')
ax.set_title('Vorhergesagt vs. tatsächlich\n(Designmatrix: $[1,\\;u_k,\\;v_k]$)')
ax.legend(fontsize=9)

plt.suptitle('Gleiche Normalengleichung — nur die Designmatrix ändert sich', fontsize=13)
plt.tight_layout()
plt.show()

---
## Zusammenfassung

| Schritt | Was passiert |
|---|---|
| **Modell wählen** | $y = mt + b$ (oder allgemeiner) |
| **Designmatrix** | Zeile $k$: $[t_k,\; 1]$ (oder andere Basis) |
| **System** | $A\hat{\mathbf{x}} \approx \mathbf{y}$ — überbestimmt |
| **Projektion** | $\mathbf{e} \perp \operatorname{Bild}(A)$ $\Rightarrow$ $A^\top \mathbf{e} = \mathbf{0}$ |
| **Normalengleichung** | $A^\top A \,\hat{\mathbf{x}} = A^\top \mathbf{y}$ |
| **Lösung** | $\hat{\mathbf{x}} = (A^\top A)^{-1} A^\top \mathbf{y}$ |

**Methode der kleinsten Fehlerquadrate = Projektion auf den Bildraum der Designmatrix.**